In [1]:
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from tqdm.auto import tqdm
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.text import TextPath
from matplotlib.patches import PathPatch
from matplotlib.transforms import Affine2D


In [2]:
# Load data
predicted_df = pd.read_csv("/Volumes/broad_dawnccle/melange/data/satmut_constitutive_effect_sizes_predicted.csv")
observed_df = pd.read_csv("/Volumes/broad_dawnccle/melange/data/satmut_constitutive_effect_sizes_observed.csv")
satmut_df = pd.concat([predicted_df, observed_df], ignore_index=True)

# reference parent sequences 
parent_seq_df = pd.read_csv("/Volumes/broad_dawnccle/melange/data/satmut_constitutive_lolliplot_reference.csv")

# Set output PATHs
output_dir = "/Volumes/broad_dawnccle/melange/figures_outputs/fig04"


In [3]:
# Map nucleotides to row indices
alphabet_dict = {'A': 0, 'C': 1, 'G': 2, 'T': 3}


# Standardize column names 
satmut_df = satmut_df.rename(columns={
    'index_offset': 'sat_ref',
    'Celltype': 'cell_type',
    'mutation_pos': 'mut_pos',
    'mutation_base': 'mut_base',
    'delta_PSI': 'post_log2Skew'  
})


satmut_df['cell_type'] = satmut_df['cell_type'].str.upper()
fasta_dict = dict(zip(parent_seq_df.iloc[:, 0], parent_seq_df.iloc[:, 1]))

# Define cell types and parent IDs
cell_types = satmut_df['cell_type'].unique()
parent_ids = sorted(satmut_df['sat_ref'].unique())

# Store results
skew_info = []

for cell_type in cell_types:
    for parent_id in tqdm(parent_ids, desc=f'Processing {cell_type}'):
        seq_family = satmut_df[
            (satmut_df['sat_ref'] == parent_id) &
            (satmut_df['cell_type'] == cell_type) &
            (satmut_df['mut_base'].isin(['A', 'C', 'G', 'T']))
        ]

        if seq_family.empty:
            continue

        ref_sequence = fasta_dict.get(f'{parent_id}')
        if ref_sequence is None:
            continue

        effect_array = np.zeros((4, len(ref_sequence)))

        for _, row in seq_family.iterrows():
            pos = int(row['mut_pos']) - 1  # 0-based
            base = row['mut_base']
            if base in alphabet_dict and 0 <= pos < len(ref_sequence):
                effect_array[alphabet_dict[base], pos] = row['post_log2Skew']

        effect_means = -effect_array.sum(axis=0) / 3.0

        skew_info.append([
            parent_id,
            cell_type,
            None, None, None,  # sat_ref_parent, var1, var2 — optional metadata
            effect_array,
            effect_means,
            ref_sequence
        ])

skew_info_df = pd.DataFrame(
    skew_info,
    columns=['ID', 'cell_type', 'sat_ref_parent', 'var1', 'var2',
             'effect_array', 'effect_means', 'ref_sequence']
)


Processing PREDICTED:   0%|          | 0/23 [00:00<?, ?it/s]

Processing OBSERVED:   0%|          | 0/23 [00:00<?, ?it/s]

In [4]:
####### z score method

# Parameters
sigma = 0.1
min_len = 2
z_thresh = 4
min_mean = 0.1
block_calls_dict = {}

for cell_type in tqdm(cell_types, desc='Identifying blocks'):
    input_df = skew_info_df[skew_info_df['cell_type'] == cell_type].reset_index(drop=True)
    block_calls_dict[cell_type] = {}

    for _, row in input_df.iterrows():
        effect_means = np.array(row['effect_means'])
        seq_id = row['ID']

        # Smooth signal
        ysmoothed = gaussian_filter1d(effect_means, sigma=sigma)

        # Compute robust stats
        q25 = np.percentile(ysmoothed, 25)
        q75 = np.percentile(ysmoothed, 75)
        median = np.median(ysmoothed)
        iqr = q75 - q25
        scale = iqr / 1.349 if iqr > 0 else 1e-6  # avoid div by zero

        # Compute robust z-scores
        z_scores = (ysmoothed - median) / scale
        
        # Mark positions above/below threshold
        is_pos_sig = z_scores > z_thresh
        is_neg_sig = z_scores < -z_thresh

        block_start_stops = []
        block_directions = []

        for is_sig, direction in [(is_pos_sig, 1), (is_neg_sig, -1)]:
            sig_idxs = np.where(is_sig)[0]

            if len(sig_idxs) == 0:
                continue

            # Group into consecutive blocks
            breaks = np.where(np.diff(sig_idxs) > 1)[0] + 1
            groups = np.split(sig_idxs, breaks)
            

            for group in groups:
                if len(group) >= min_len:
                    start, stop = group[0], group[-1] + 1
                    window_len = stop - start
                    window_mean = effect_means[start:stop].mean()

                    
                    if direction == 1 and window_mean >= min_mean:
                        block_start_stops.append((start, stop))
                        block_directions.append(direction)
                    elif direction == -1 and window_mean <= -min_mean:
                        block_start_stops.append((start, stop))
                        block_directions.append(direction)


        block_calls_dict[cell_type][seq_id] = (block_start_stops, block_directions)


Identifying blocks:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# plotting helpers

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# Color and base mapping
alphabet_dict = {'A': 0, 'C': 1, 'G': 2, 'T': 3, 'U': 3}
colors = ['g', 'b', 'C1', 'r']
markerfmts = [c + 'o' for c in colors]


# Main plotter
def plot_simple_lollipop(seq_id, cell_type, skew_info_df, block_calls_dict, save_dir=None):
    # Get data
    row = skew_info_df[(skew_info_df['ID'] == seq_id) & (skew_info_df['cell_type'] == cell_type)].iloc[0]
    effect_array = row['effect_array']
    effect_means = row['effect_means']
    ref_seq = row['ref_sequence']
    fig, ax = plt.subplots(figsize=(6.5, 1.5))

    locs = np.arange(effect_array.shape[1])
    for i in range(4):
        heads = np.array(effect_array[i, :], copy=True)
        heads[heads == 0.] = np.nan
        markerline, stemlines, baseline = ax.stem(locs + 0.5, heads, colors[i], markerfmt=markerfmts[i], basefmt=' ')
        plt.setp(stemlines, 'linestyle', '-', 'linewidth', 0.5, 'alpha', 1)
        plt.setp(markerline, 'markersize', 1, 'alpha', 1)
        plt.setp(baseline, 'linewidth', 0)


    # Draw reference logo without blocks (show full sequence)
    draw_reference_logo(ax, ref_seq, effect_means, block_calls_dict[cell_type][seq_id])

    # Axis formatting
    xtick_positions = np.arange(0, len(ref_seq), 10)
    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(xtick_positions, fontsize=6, fontname='Helvetica')
    ax.set_yticks([1, 0.5, 0, -0.5, -1])
    ax.set_ylabel('Effect', fontsize=6, fontname='Helvetica')
    ax.set_ylim(-1, 1)
    ax.set_title(f'{seq_id}  {cell_type}', fontsize=6, fontname='Helvetica', y=1.05)

    # Clean axes
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.spines['left'].set_linewidth(0.5)
    ax.tick_params(width=0.5, labelsize=6)
    plt.xticks(fontsize=6, fontname='Helvetica')
    plt.yticks(fontsize=6, fontname='Helvetica')

    # Save figure
    if save_dir:
        import os
        os.makedirs(save_dir, exist_ok=True)
        fname = f'{seq_id}_{cell_type}.pdf'
        fname = 'fig04_' + fname
        path = os.path.join(save_dir, fname)
        plt.savefig(path, format='pdf', bbox_inches='tight')
        plt.close(fig)



def draw_reference_logo(ax, ref_seq, effect_means, blocks=None, baseline=0.0, scale_factor=1.0, vpad=0.05):
    print("Blocks for", seq_id, ":", blocks)

    for i, base in enumerate(ref_seq):
        # Replace T with U
        base = 'U' if base == 'T' else base
        if base not in alphabet_dict:
            continue

        val = effect_means[i]
        if np.isnan(val):
            continue

        # Only draw if within any block (if blocks specified)
        if blocks:
            in_block = any(start <= i < stop for (start, stop) in blocks[0])
            if not in_block:
                continue

        height = abs(val) * scale_factor
        direction = np.sign(val)

        # Choose color (updated for 'U')
        base_color = {'A': 'green', 'C': 'blue', 'G': 'orange', 'U': 'red'}[base]

        # Set vertical position
        y_pos = baseline + vpad if direction >= 0 else baseline - height - vpad

        # Draw the base letter
        tp = TextPath((0, 0), base, size=1, prop=dict(weight='bold'))
        trans = Affine2D().scale(1, height).translate(i + 0.5, y_pos)
        patch = PathPatch(tp, transform=trans + ax.transData, color=base_color, lw=0)
        ax.add_patch(patch)

    return ax


In [6]:
# Plot sequences

cell_types = skew_info_df['cell_type'].unique()

for cell_type in tqdm(cell_types, desc="Plotting by cell type"):
    # all unique sequence IDs for this cell type
    seq_ids = skew_info_df.loc[skew_info_df['cell_type'] == cell_type, 'ID'].unique()

    for seq_id in seq_ids:
        try:
            plot_simple_lollipop(
                seq_id=seq_id,
                cell_type=cell_type,
                skew_info_df=skew_info_df,
                block_calls_dict=block_calls_dict,
                save_dir=output_dir
            )
        except Exception as e:
            print(f"Failed to plot {seq_id} ({cell_type}): {e}")


Plotting by cell type:   0%|          | 0/2 [00:00<?, ?it/s]

Blocks for ENSG00000028203.18;VEZT;chr12-95258244-95258282-95257149-95257239-95262905-95263081__0:0:0 : ([(134, 136)], [-1])


1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
Zapf NOT subset; don't know how to subset; dropped
feat NOT subset; don't know how to subset; dropped
morx NOT subset; don't know how to subset; dropped
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
Zapf NOT subset; don't know how to subset; dropped
feat NOT subset; don't know how to subset; dropped
morx NOT subset; don't know how to subset; dropped


Blocks for ENSG00000089006.17;SNX5;chr20-17944121-17944144-17943109-17943195-17947485-17947621__0:0:0 : ([(134, 136)], [-1])
Blocks for ENSG00000104695.13;PPP2CB;chr8-30799716-30799755-30797580-30797754-30812319-30812597__0:0:0 : ([(118, 120)], [-1])


1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
Zapf NOT subset; don't know how to subset; dropped
feat NOT subset; don't know how to subset; dropped
morx NOT subset; don't know how to subset; dropped
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
Zapf NOT subset; don't know how to subset; dropped
feat NOT subset; don't know how to subset; dropped
morx NOT subset; don't know how to subset; dropped


...

Blocks for ENSG00000204308.8;RNF5;chr6-32180008-32180126-32179876-32179931-32180228-32180793__0:0:0 : ([(57, 66), (184, 189)], [1, 1])
Blocks for ENSG00000215018.10;COL28A1;chr7-7515813-7515840-7511090-7511135-7517795-7517837__0:0:0 : ([], [])


1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
Zapf NOT subset; don't know how to subset; dropped
feat NOT subset; don't know how to subset; dropped
morx NOT subset; don't know how to subset; dropped
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
Zapf NOT subset; don't know how to subset; dropped
feat NOT subset; don't know how to subset; dropped
morx NOT subset; don't know how to subset; dropped


Blocks for ENSG00000229140.11;CCDC26;chr8-128828063-128828094-128827408-128827701-129019397-129019430__0:0:0 : ([], [])
Blocks for ENSG00000257341.5;AL928654.7;chr14-105488330-105488388-105488165-105488260-105488470-105488516__0:0:0 : ([(28, 30), (31, 33), (47, 50), (108, 111)], [1, 1, 1, 1])


1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
Zapf NOT subset; don't know how to subset; dropped
feat NOT subset; don't know how to subset; dropped
morx NOT subset; don't know how to subset; dropped
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
Zapf NOT subset; don't know how to subset; dropped
feat NOT subset; don't know how to subset; dropped
morx NOT subset; don't know how to subset; dropped


Blocks for ENSG00000286034.1;RP11-654O1.1;chr8-128324908-128325005-128323130-128323188-128336131-128336213__0:0:0 : ([(65, 67), (70, 72), (73, 76), (173, 177)], [1, 1, 1, 1])
